# <u>Revisiting the dynamical properties of the Pedlosky's Two-Layer Model for Finite Amplitude Baroclinic Waves </u>

# <i>Project notebook</i>

We begin this notebook by importing the necessary packages and modules.

In [ ]:
import sys, os
import glob
sys.path.extend([os.path.abspath('../')])

from PedloskySystem import System

import numpy as np
from numpy import linalg as la 
import matplotlib.pyplot as plt
import numpy.random as rnd
from joblib import Parallel, delayed

from auto2.diagrams.bifurcations import BifurcationDiagram
from auto2.parsers.config import ConfigParser

The following functions will be useful throughout the notebook.

In [ ]:
def two_plots(ax1_data, ax2_data, size=(10, 5), ax1_label=[r'$T$', r'$A(T)$'], ax2_label=[r'$A$', r'$B$']):
    """
    Function to create two plots side by side with specified data and labels (used for visualization purposes).

    Parameters
    ----------
    ax1_data: list
        Data for the first plot, should be a list of two arrays [x_data, y_data]

    ax2_data: list
        Data for the second plot, should be a list of two arrays [x_data, y_data]

    size: tuple, optional
        Size of the figure, default is (10, 5)

    ax1_label: list, optional
        Labels for the axes of the first plot, default is [r'$T$', r'$A(T)$']

    ax2_label: list, optional
        Labels for the axes of the second plot, default is [r'$A$', r'$B$']

    Return
    ------
    None
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize = size)
    fig.tight_layout(pad=5.0)

    ax1.set_xlabel(ax1_label[0], fontsize=12)
    ax1.set_ylabel(ax1_label[1], fontsize=12)

    ax2.set_xlabel(ax2_label[0], fontsize=12)
    ax2.set_ylabel(ax2_label[1], fontsize=12)

    ax1.grid(True, linestyle='-.', which='both')
    ax1.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)
    ax2.grid(True, linestyle='-.', which='both')
    ax2.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)

    ax1.plot(ax1_data[0], ax1_data[1], color='cornflowerblue')

    ax2.plot(ax2_data[0], ax2_data[1], color='lightcoral')

    plt.show()

In [ ]:
def find_periodic_orbit(kc, gamma, a = np.pi*np.sqrt(2), m = 1.0, dt_periodic = 1.e-3, t_min_periodic = 1000, t_max_periodic = 500, t_max_transient = 1.e5, epsilon = 0.1, parallel = True, save = True, path = None):
    """"
    This function is used to extract a periodic orbit from a long-time integration of the system for a given value of gamma. It will be used to find initial periodic orbits for the continuation with AUTO2.
    The function outputs a plot of the periodic orbit for visual inspection. If the function fails to extract a periodic orbit, the user might increase the value of t_min_periodic and reduce the value of epsilon.

    Parameters
    ----------
    kc: int
        Number of Fourier modes in the x direction
        
    gamma: float
        Value of the parameter gamma

    a: float, optional
        Sum of squared x and y Fourier modes, default is np.pi * np.sqrt(2)

    m: float, optional
        y Fourier mode, default is 1.0

    dt_periodic: float, optiona
        Time granularity of the saved states along the periodic orbit, default is 1.e-3

    t_min_periodic: int, optional
        Minimum time index to start looking for periodicity, default is 1000

    t_max_periodic: int, optional
        Maximum time index to find periodicity, default is 500

    t_max_transient: float, optional
        Integration time during the transient phase to converge to a periodic orbit, default is 1.e5

    epsilon: float, optional
        Tolerance to determine if the orbit is periodic, default is 0.1

    parallel: bool, optional
        Whether to compute the trajectories in parallel or not

    save: bool, optional
        Whether to save the periodic orbit in a .dat file or not, default is True

    path: str, optional
        Path to the directory where the files will be saved, if None, files will be saved in the "Data" directory

    Return
    ------
    filename: str
        Name of the file where the periodic orbit is saved. AUTO2 will use this file as input

    parameters: dict
        Dictionary containing the parameters of the system

    Example
    -------
    See below for an example of usage.
    """
    ds = System(kc, gamma, a, m)

    parameters = {
        'gamma': gamma,
        'a': a,
        'm': m}

    A0 = 1. #np.random.rand() * 10 - 5 for a random initial condition
    B0 = 1. #np.random.rand() * 10 - 5 for a random initial condition
    X0 = np.array([np.concatenate(([A0, B0], - np.ones(kc) * A0 ** 2))])

    # First transient integration to get closer to the attractor
    t_max = t_max_transient
    time_integration = np.linspace(0, t_max, 3)  # no need to store many points during the transient
    solution = ds.integration_system(time_integration, X0, parallel=parallel)

    #two_plots([time_integration, solution[0, :, 0]], [solution[0, :, 0], solution[0, :, 1]], size = (12, 6))

    # Second precise integration to plot the attractor
    t_max = t_max_periodic
    dt = dt_periodic
    time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)
    X0 = np.array([solution[0, - 1]])
    solution = ds.integration_system(time_integration, X0, parallel=parallel)

    for i in range(t_min_periodic, solution.shape[1]): # The starting index must be cautiousy chosen
        if np.linalg.norm(solution[0:, i] - solution[0, 0]) < epsilon:
            break

    for j in range(i, solution.shape[1]):
        if np.linalg.norm(solution[0][j] - solution[0][0]) > np.linalg.norm(solution[0][j - 1] - solution[0][0]):
            break

    two_plots([time_integration[:j], solution[0, :j, 0]], [solution[0, :j, 0], solution[0, :j, 1]])

    print("Value of gamma: ", gamma)
    print("Period: ", time_integration[j - 1])

    saved_solution = np.zeros((solution.shape[- 1] + 1, j))
    saved_solution[1 :, :] = solution[0, : j].T # Transpose to have the right shape
    saved_solution[0] = time_integration[: j] # First row is the time grid
    saved_solution[1 :, - 1] = solution[0, 0] # To close the trajectory

    if save:
        if path is None:
            np.savetxt(f'Data/Periodic_Orbit_{gamma}.dat', saved_solution.T)
            return f'Data/Periodic_Orbit_{gamma}.dat', parameters
        else:
            np.savetxt(f'{path}/Periodic_Orbit_{gamma}.dat', saved_solution.T)
            return f'{path}/Periodic_Orbit_{gamma}.dat', parameters
        
    return None, parameters

In [ ]:
def find_period_1d(t, X, eps=1e-4, skip=1000):
    """"
    This is a minimal function used to find the period of a 1D time series.

    Parameters
    ----------
    t: numpy.ndarray
        Time grid of the integration

    X: numpy.ndarray
        1D time series resulting from the integration

    eps: float, optional
        Tolerance to determine if the orbit is periodic, default is 1e-4

    skip: int, optional
        Number of initial time steps to skip before looking for periodicity, default is 1000

    Return
    ------
    index: int
        Index of the time grid where the period is found

    period: float
        Value of the period found

    Example
    -------
    See below for an example of usage.
    """
    X0 = X[0]
    dX0 = X[1] - X[0]

    for i in range(skip, len(X) - 1):
        if abs(X[i] - X0) < eps:
            dXi = X[i + 1] - X[i]
            if dX0 * dXi > 0:  
                return i, t[i] - t[0]

    raise RuntimeError("No period found")


In [ ]:
def analyze_orbit(X, period, dt):
    """"
    Function to compute the (discretized) L2-norm of a periodic orbit, given the time series of the orbit and its period.

    Parameters
    ----------
    X: numpy.ndarray
        Time series of the periodic orbit

    period: float
        Period of the orbit

    dt: float
        Time step of the integration

    Return
    ------
    norm: float
        L2-norm of the periodic orbit

    Example
    -------
    See below for an example of usage.
    """
    norm_squared = np.sum(X ** 2) * dt / period
    norm = np.sqrt(norm_squared)

    return norm

In [ ]:
def export_branches(branch, branch_keys, translate, path = None):
    """"
    Function to extract the data of the branches computed with AUTO2 and save them in text files for further analysis and visualization. The function also extracts the data of the period-doubling bifurcation points if they are detected.
    Note that it is the responsibility of the user to save the data in the directory of their choice and to create this directory if it does not exist.

    Parameters
    ----------
    branch: auto2.diagrams.bifurcations.BifurcationDiagram
        Bifurcation diagram object containing the branches computed with AUTO2

    branch_keys: list, optional
        List of branch keys to export, if None, all branches will be exported

    translate: int
        Integer to add to the branch keys when saving the files, useful if the user wants to save the branches in the same directory with different names

    path: str, optional
        Path to the directory where the files will be saved, if None, files will be saved in the "Data" directory

    Return
    ------
    None

    Example
    -------
    See below for an example of usage.
    """

    po_keys = list(branch.po_branches.keys())
    print("Branch keys: ", po_keys)
    number_bp = len(po_keys)
    print("Number of branch detected: ", number_bp)

    if branch_keys is not None:
        po_keys = branch_keys
        print("Branch keys after filtering: ", po_keys)
        print("Saving branches: ", po_keys)

    for i in po_keys:

        hp = branch.po_branches[i]["continuation"]
        sols = hp.full_solutions_list

        pd_search = hp.get_solution_by_label('PD')
        result_pd = []

        if pd_search == []:
            print("No PD point found in branch ", i)
        else:
            for x in pd_search:
                result_pd.append([x["gamma"], x["MAX A"], x["L2-NORM"]])


        result_branch = []

        for j, sol in enumerate(sols):
            gamma_value = sol["gamma"]
            max_A = sol["MAX A"]
            norm_L2 = sol["L2-NORM"]
            T_period = sol["T"]

            get_Floquet = hp.orbit_stability(sols[j]["PT"])
            if type(get_Floquet) is list:
                get_Floquet = np.array(get_Floquet)
                check_overflow = np.any(get_Floquet.flatten() > 1e10) 
                if not check_overflow:
                    test_Floquet = np.sqrt(get_Floquet[:, 0] ** 2 + get_Floquet[:,1] ** 2)
                    stability = int(not np.any(test_Floquet > 1 + 1e-3)) # If True, orbit is stable
                else:
                    stability = int(False)
            
            else:
                stability = -1

            result_branch.append((gamma_value, max_A, norm_L2, T_period, stability))

        if path is None:
            name = f"Data/branch_{i + translate}.txt"
        else:
            name = os.path.join(path, f"branch_{i + translate}.txt")

        np.savetxt(name, result_branch, header="gamma max_A L2-norm T Stability", fmt="%.6f")
        print(f"Saved {name}")

        if path is None:
            name_pd = f"Data/branch_{i + translate}_PD_points.txt"
        else:
            name_pd = os.path.join(path, f"branch_{i + translate}_PD_points.txt")
            
        if pd_search != []:
            np.savetxt(name_pd, result_pd, header="gamma max_A L2-norm", fmt="%.6f")
            print(f"Saved {name_pd}")

In [ ]:
def lyapunov_gamma(gamma: float, ic_system: np.ndarray, s: int = 0.1, steps: float = 5 * 10 ** 4) -> np.ndarray:
    """"
    This function computes the Lyapunov spectrum of the Pedlosky system for a given value of gamma and a set of initial conditions.
    While the function may seem redundant, it is used for parallel computations of the Lyapunov spectrum for different values of gamma.

    Parameters
    ----------
    gamma: float
        Value of the parameter gamma

    ic_system: numpy.ndarray (of shape (n_ic, kc + 2))
        Initial conditions of the system. The 'n_ic' state vectors should be obtained after a transient integration

    s: float
        Step size for the integration of the variational equations

    steps: float
        Number of steps for integrating the variational equations

    Return
    ------
    spectrum_Lya: numpy.ndarray (of shape (n_ic, self._kc + 2,))
        Lyapunov spectrum of the Pedlosky system for the given value of gamma and set of initial conditions

    Example
    -------
    >>> kc = 1
    >>> gamma = 0.5
    >>> a = np.pi * np.sqrt(2)
    >>> m = 1
    >>> ds = System(kc, gamma, a, m)
    >>> X0 = np.array([[1, -1.5, -1], [0.5, 0.5, -0.25], [-1, 1, -1], [2, -1, -4]])
    >>> s = 0.01
    >>> steps = 10000
    >>> print(lyapunov_gamma(gamma, X0, s, steps))
    [[-0.0174137  -0.1055322  -0.79372121]
     [-0.03931313 -0.02656701 -0.85078703]
     [ 0.00974479 -0.05677824 -0.86963377]
     [-0.02966311 -0.03854973 -0.84845423]]
    """
    ds = System(1, gamma, np.pi * np.sqrt(2), 1)
    steps = int(steps)
    spectrum_Lya = ds.get_Lyapunov_spectrum(ic_system, s, steps)

    return spectrum_Lya

We start by creating (and cleaning if it exists) the "Data" folder. If the folder already exists, an OSError is raised.

In [ ]:
os.makedirs("Data", exist_ok=False)
os.makedirs("Data/toy_model", exist_ok=False)
os.makedirs("Data/full_model", exist_ok=False)

for f in glob.glob(os.path.join("Data/toy_model", "*")):
    os.remove(f)    

for f in glob.glob(os.path.join("Data/full_model", "*")):
    os.remove(f)

print("All files cleared in", "Data")
print("All files cleared in", "Data/toy_model")
print("All files cleared in", "Data/full_model")

For practical purposes, we hide plots. The following command, <code>%matplotlib agg </code> switches matplotlib to a non-GUI. It can be re-enabled by replacing the line by <code>%matplotlib inline</code>/

In [ ]:
%matplotlib inline

### <span style="color:gray"><i>Integrating the model for fun: basic exploration</i></span>

In [ ]:
# Parameters
kc = 5
gamma = 0.1
a = np.pi * np.sqrt(2)
m = 1

# Initial condition construction
A0 = 1.
B0 = -1.

X0 = np.empty((1, kc + 2))
X0[0][0] = A0
X0[0][1] = B0

for i in range(2, len(X0[0])):
    X0[0][i] = - A0 ** 2

# Time integration
tMax = 10 ** 3
dt = 10 ** (- 1)
time_integration = np.linspace(0, tMax, int(tMax / dt))   

ds = System(kc, gamma, a, m)
solution = ds.integration_system(time_integration, X0)

two_plots([time_integration, solution[0, :, 0]], [solution[0, :, 0], solution[0, :, 1]], size = (12, 6))

If needed, the solution can be saved for later use with the following code (change the directory accordingly):

In [ ]:
save_solution = False

if save_solution:
    sol_list = solution[0, :, :]
    saved_solution = np.zeros((sol_list.shape[- 1] + 1, sol_list.shape[0]))
    saved_solution[1 :, :] = sol_list.T # Transpose to have the right shape
    saved_solution[0] = time_integration # First row is the time grid
    np.savetxt(f'Data/solution_{gamma}.dat', saved_solution.T)

In [ ]:
gamma = 0.1
filename, parameters = find_periodic_orbit(kc, gamma, save = False)

## <span style="color:orange">1.  Pedlosky's model</span>

<span><i>In this section, we verify the results of Section II of the paper, in particular Sections II.C, where we study the effect of the cut-off $k_c$ on the model's solutions, and II.E, where we examine the Hamiltonian case corresponding to $\gamma=0$.</i></span>

### <span style="color:gray"><i>Effect of the truncation</i></span>

Solution of type 1: fixed points.

In [ ]:
kc_list = np.linspace(25, 250, 10, dtype=int)
A_long_list = []
phi_long_list = []

for kc in kc_list:

    print("Computing for kc = ", kc)
    ds = System(kc, 1.0, np.pi * np.sqrt(2), 1)

    A0 = 1.
    B0 = 0. 
    X0 = np.array([np.concatenate(([A0, B0], - np.ones(kc) * A0 ** 2))])

    # First transient integration to get closer to the attractor
    t_max = 10 ** 5
    dt = 10 ** (- 1)
    time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)
    solution = ds.integration_system(time_integration, X0)
    phi_list = ds.Phi(1.0, solution)

    A_long_list.append(solution[0, - 1, 0])
    phi_long_list.append(phi_list[- 1])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(kc_list, A_long_list - np.ones(len(A_long_list)), color='mediumseagreen')
ax.set_xlabel(r'$k_c$', fontsize=12)
ax.set_ylabel(r'$A(\infty)-1$', fontsize=12)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(kc_list, phi_long_list, color='mediumseagreen')
ax.set_xlabel(r'$k_c$', fontsize=12)
ax.set_ylabel(r'$\phi(\infty)$', fontsize=12)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)
plt.show()

Solution of type 2: periodic orbits.

In [ ]:
kc_list = np.arange(1, 30 + 1)
period_A_list = []
norm_A_list = []
period_phi_list = []
norm_phi_list = []

for kc in kc_list:

    print("Computing for kc = ", kc)
    ds = System(kc, 0.05, np.pi * np.sqrt(2), 1)

    A0 = 1.
    B0 = 0. 
    X0 = np.array([np.concatenate(([A0, B0], - np.ones(kc) * A0 ** 2))])

    # First transient integration to get closer to the attractor
    t_max = 10 ** 5
    dt = 10 ** (- 2)
    time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)
    solution = ds.integration_system(time_integration, X0)

    # Second precise integration to plot the attractor
    t_max = 500
    dt = 10 ** (-3)
    time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)
    X0 = np.array([solution[0, - 1]])
    solution = ds.integration_system(time_integration, X0)

    idx_A, period_A = find_period_1d(time_integration, solution[0, :, 0])
    orbit_A = solution[0, : idx_A, 0]
    period_A_list.append(period_A)

    phi_list = ds.Phi(1.0, solution)
    idx_phi, period_phi = find_period_1d(time_integration, phi_list)
    orbit_phi = phi_list[: idx_phi]
    period_phi_list.append(period_phi)

    dt = time_integration[1] - time_integration[0]
    norm_A = analyze_orbit(orbit_A, period_A, dt)
    norm_phi = analyze_orbit(orbit_phi, period_phi, dt)
    norm_A_list.append(norm_A)
    norm_phi_list.append(norm_phi)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(kc_list, norm_A_list, color='mediumseagreen')
ax.set_xlabel(r'$k_c$', fontsize=12)
ax.set_ylabel(r'$\parallel A \parallel$', fontsize=12)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(kc_list, norm_phi_list, color='mediumseagreen')
ax.set_xlabel(r'$k_c$', fontsize=12)
ax.set_ylabel(r'$\parallel \Phi(y=1) \parallel$', fontsize=12)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)
plt.show()

### <span style="color:gray"><i>Hamiltonian system</i></span>

In [ ]:
kc = 10
gamma = 0
a = np.pi * np.sqrt(2)
m = 1

A0 = 1. # E < 0
B0 = 0. # E < 0
X0 = np.empty((1, kc + 2))
X0[0][0] = A0
X0[0][1] = B0

for i in range(2, len(X0[0])):
    X0[0][i] = - A0 ** 2

# Time integration
tMax = 10 ** 2
dt = 10 ** (-4)
time_integration = np.linspace(0, tMax, int(tMax / dt))   

ds = System(kc, gamma, a, m)
solution = ds.integration_system(time_integration, X0)

H_value = ds.Hamiltonian(solution)
print(H_value[0])

In [ ]:
fig, ax = plt.subplots(figsize = (6, 5))

ax.plot(solution[0, :, 0], solution[0, :, 1], color='mediumseagreen')
ax.set_xlabel(r'$A$', fontsize=12)
ax.set_ylabel(r'$B$', fontsize=12)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)
ax.set_title(r'$E<0$', fontsize=14)

plt.show()

In [ ]:
A0 = 1. # E > 0
B0 = 1.5 # E > 0
X0 = np.empty((1, kc + 2))
X0[0][0] = A0
X0[0][1] = B0

for i in range(2, len(X0[0])):
    X0[0][i] = - A0 ** 2

# Time integration
tMax = 10 ** 2
dt = 10 ** (-4)
time_integration = np.linspace(0, tMax, int(tMax / dt))   

ds = System(kc, gamma, a, m)
solution = ds.integration_system(time_integration, X0)

H_value = ds.Hamiltonian(solution)
print(H_value[0])

In [ ]:
fig, ax = plt.subplots(figsize = (6, 5))

ax.plot(solution[0, :, 0], solution[0, :, 1], color='mediumseagreen')
ax.set_xlabel(r'$A$', fontsize=12)
ax.set_ylabel(r'$B$', fontsize=12)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)
ax.set_title(r'$E>0$', fontsize=14)

plt.show()

Energy is not exactly conserved due to numerical errors, but it should be close to the initial value.

In [ ]:
fig, ax = plt.subplots(figsize = (6, 5))

ax.plot(time_integration, H_value, color='mediumseagreen')
ax.set_xlabel(r'$t$', fontsize=12)
ax.set_ylabel(r'$H$', fontsize=12)
ax.grid(True, linestyle='-.', which='both')
ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)

plt.show()

## <span style="color:orange">2.  Linear stability analysis of the model</span>

<span><i>Wwe now provide the code to verify our stability analysis of the fixed points of the system, supporting the main text of Section III of the paper..</i></span>

<span><i>We now study the toy model, defined as the full Pedlosky model with $k_c = 1$. We provide the code for all the nonlinear analyses, including the basins of attraction and the continuation of periodic orbits with auto-AUTO. The Lyapunov exponents are presented in the last section of this notebook.</i></span>

### <span style="color:gray"><i>Stability analysis of the equilibrium points ($A^2=1$)</i></span>

Here, we take $A^2=1$ independently of the value of the truncation. See next subsection for an exact treatment.

Out of consistency, we verify that the Jacobian matrices at the equilibrium points $\bm{X}_\pm$ are similar.

In [ ]:
for m in rnd.randint(1, 10, 50):
    for a in 10 * rnd.rand(50):

        ds = System(10, 1, a, m)
        XP, XM = ds.infinite_equilibrium_points()
        DFP = ds.Jacobian(XP)
        DFM = ds.Jacobian(XM)
        P = - np.diag(np.ones(12))
        P[0, 0] = 1
        P[1, 1] = 1

        if np.allclose(DFM, P @ DFP @ la.inv(P)) == False:
            print("At a = ", a, " and m = ", m, "DFP and DFM are not similar")

In [ ]:
kc = 10
a = np.pi * np.sqrt(2)
m = 1

gamma_list = np.arange(5, 0, -0.001)
eigenvalues = np.empty((len(gamma_list), 3, kc + 2), dtype=complex)

for i, g in enumerate(gamma_list):

    ds = System(kc, g, a, m) 
    X0 = np.zeros(kc + 2)
    XP, XM = ds.infinite_equilibrium_points() # Computes the exact equilibrium in the kc -> infinity limit
    DF0 = ds.Jacobian(X0) 
    DFP = ds.Jacobian(XP)
    DFM = ds.Jacobian(XM)

    eigenavalues_O = la.eigvals(DF0)
    eigenavalues_P = la.eigvals(DFP)
    eigenavalues_M = la.eigvals(DFM)
    eigenvalues[i] = np.array([eigenavalues_O, eigenavalues_P, eigenavalues_M])

    if not np.allclose(eigenavalues_M, eigenavalues_P):
        print("At gamma = ", g, "eigenvalues are not the same for DFP and DFM")

In [ ]:
count_positive = np.empty(len(gamma_list))

for i in range(len(gamma_list)):
    count_positive[i] = len(np.where(eigenvalues[i, 0] > 0)[0])

if np.allclose(count_positive, np.ones(len(gamma_list))) is True:
    print("At least one eigenvalue associated to the equilibrium point O has a positive real part for all values of gamma in the list.")

In [ ]:
is_complex = np.empty(len(gamma_list), dtype=bool)
count_positive = np.empty(len(gamma_list))

for i in range(len(gamma_list)):
    is_complex[i] = np.any(np.iscomplex(eigenvalues[i, 1]))
    count_positive[i] = len(np.where(eigenvalues[i, 1].real > 0)[0])

if np.allclose(count_positive, np.zeros(len(gamma_list))) is True:
    print("All eigenvalues associated to the equilibrium point P have negative real part for all values of gamma in the list.")

In [ ]:
is_complex = np.empty(len(gamma_list), dtype=bool)
count_positive = np.empty(len(gamma_list))

for i in range(len(gamma_list)):
    is_complex[i] = np.any(np.iscomplex(eigenvalues[i, 2]))
    count_positive[i] = len(np.where(eigenvalues[i, 2].real > 0)[0])

if np.allclose(count_positive, np.zeros(len(gamma_list))) is True:
    print("All eigenvalues associated to the equilibrium point M have negative real part for all values of gamma in the list.")

In [ ]:
not_positive_where = np.where(count_positive != 0)[0] # This is used in the cases where kc = 1 and kc = 2
print(gamma_list[not_positive_where])

In [ ]:
print(gamma_list[np.where(is_complex)[0]][0]) # Value of \tilde{gamma} where the eigenvalues associated to the equilibrium point P become complex conjugate pairs

How $\widetilde{\gamma}$ depends on $k_c$?

In [ ]:
kc_max = 70
a = np.pi * np.sqrt(2)
m = 1

gamma_list = np.arange(3.9, 3.8, -0.0001)
kc_list = np.arange(10, kc_max + 1, 10)
print(kc_list)

is_complex = np.empty(len(gamma_list), dtype=bool)
gamma_critical = np.zeros(len(kc_list))

for i, kc in enumerate(kc_list):
    for j, g in enumerate(gamma_list):

        ds = System(kc, g, a, m) # Creates an instance of the system

        XP = ds.infinite_equilibrium_points()[0]
        DFP = ds.Jacobian(XP)
        eigenavalues_P = la.eigvals(DFP)

        is_complex[j] = np.any(np.iscomplex(eigenavalues_P))

    gamma_critical[i] = gamma_list[np.where(is_complex)[0]][0]

In [ ]:
print(gamma_critical)

### <span style="color:gray"><i>Stability analysis of the equilibrium points (exact solution for $A^2$)</i></span>

Here, we take th exact value for $A^2$ which depends on the value of the truncation. See Eq. (29) of the paper.

In [ ]:
for m in rnd.randint(1, 10, 50):
    for a in 10 * rnd.rand(50):

        ds = System(10, 1, a, m)
        XP, XM = ds.exact_equilibrium_points()
        DFP = ds.Jacobian(XP)
        DFM = ds.Jacobian(XM)
        P = - np.diag(np.ones(12))
        P[0, 0] = 1
        P[1, 1] = 1

        if np.allclose(DFM, P @ DFP @ la.inv(P)) == False:
            print("At a = ", a, " and m = ", m, "DFP and DFM are not similar")

In [ ]:
kc = 10
a = np.pi * np.sqrt(2)
m = 1

gamma_list = np.arange(5, 0, -0.001)
eigenvalues = np.empty((len(gamma_list), 3, kc + 2), dtype=complex)

for i, g in enumerate(gamma_list):

    ds = System(kc, g, a, m)
    X0 = np.zeros(kc + 2)
    XP, XM = ds.exact_equilibrium_points() # Computes the exact equilibrium
    DF0 = ds.Jacobian(X0) 
    DFP = ds.Jacobian(XP)
    DFM = ds.Jacobian(XM)

    eigenavalues_O = la.eigvals(DF0)
    eigenavalues_P = la.eigvals(DFP)
    eigenavalues_M = la.eigvals(DFM)
    eigenvalues[i] = np.array([eigenavalues_O, eigenavalues_P, eigenavalues_M])

    if not np.allclose(eigenavalues_M, eigenavalues_P):
        print("At gamma = ", g, "eigenvalues are not the same for DFP and DFM")

In [ ]:
count_positive = np.empty(len(gamma_list))

for i in range(len(gamma_list)):
    count_positive[i] = len(np.where(eigenvalues[i, 0] > 0)[0])

if np.allclose(count_positive, np.ones(len(gamma_list))) is True:
    print("At least one eigenvalue associated to the equilibrium point O has a positive real part for all values of gamma in the list.")

In [ ]:
is_complex = np.empty(len(gamma_list), dtype=bool)
count_positive = np.empty(len(gamma_list))

for i in range(len(gamma_list)):
    is_complex[i] = np.any(np.iscomplex(eigenvalues[i, 1]))
    count_positive[i] = len(np.where(eigenvalues[i, 1].real > 0)[0])

if np.allclose(count_positive, np.zeros(len(gamma_list))) is True:
    print("All eigenvalues associated to the equilibrium point P have negative real part for all values of gamma in the list.")

In [ ]:
not_positive_where = np.where(count_positive != 0)[0] # This is used in the cases where kc = 1 and kc = 2
print(gamma_list[not_positive_where])

In [ ]:
print(gamma_list[np.where(is_complex)[0]][0])

In [ ]:
is_complex = np.empty(len(gamma_list), dtype=bool)
count_positive = np.empty(len(gamma_list))

for i in range(len(gamma_list)):
    is_complex[i] = np.any(np.iscomplex(eigenvalues[i, 2]))
    count_positive[i] = len(np.where(eigenvalues[i, 2].real > 0)[0])

np.allclose(count_positive, np.zeros(len(gamma_list)))

In [ ]:
print(gamma_list[np.where(is_complex)[0]][0]) # Value of \tilde{gamma} where the eigenvalues associated to the equilibrium point P become complex conjugate pairs. Compare with the value obtained in the infinite kc limit

In [ ]:
kc_max = 70
a = np.pi * np.sqrt(2)
m = 1

gamma_list = np.arange(3.9, 3.8, -0.0001)
kc_list = np.arange(10, kc_max + 1, 10)
print(kc_list)

is_complex = np.empty(len(gamma_list), dtype=bool)
gamma_critical = np.zeros(len(kc_list))

for i, kc in enumerate(kc_list):
    for j, g in enumerate(gamma_list):

        ds = System(kc, g, a, m)

        XP = ds.exact_equilibrium_points()[0]
        DFP = ds.Jacobian(XP)
        eigenavalues_P = la.eigvals(DFP)

        is_complex[j] = np.any(np.iscomplex(eigenavalues_P))

    gamma_critical[i] = gamma_list[np.where(is_complex)[0]][0]

In [ ]:
print(gamma_critical)

## <span style="color:orange">3.  Toy model</span>

<span><i>We now study the toy model, defined as the full Pedlosky model with $k_c = 1$. We provide the code for all the nonlinear analyses, including the basins of attraction and the continuation of periodic orbits with auto-AUTO. The Lyapunov exponents are presented in the last section of this notebook.</i></span>

### <span style="color:gray"><i>Stability analysis of the equilibrium points (exact solution for $A^2$)</i></span>

In [ ]:
kc = 1
a = np.pi * np.sqrt(2)
m = 1

gamma_list = np.arange(5, 0, -0.001)
eigenvalues = np.empty((len(gamma_list), 3, kc + 2), dtype=complex)

for i, g in enumerate(gamma_list):

    ds = System(kc, g, a, m) # Creates an instance of the system
    X0 = np.zeros(kc + 2)
    XP, XM = ds.exact_equilibrium_points() # Computes the exact equilibrium
    DF0 = ds.Jacobian(X0) 
    DFP = ds.Jacobian(XP)
    DFM = ds.Jacobian(XM)

    eigenavalues_O = la.eigvals(DF0)
    eigenavalues_P = la.eigvals(DFP)
    eigenavalues_M = la.eigvals(DFM)
    eigenvalues[i] = np.array([eigenavalues_O, eigenavalues_P, eigenavalues_M])

    if not np.allclose(eigenavalues_M, eigenavalues_P): # Sanity check (it was already verified earlier that DFP and DFM are similar)
        print("At gamma = ", g, "eigenvalues are not the same for DFP and DFM")

In [ ]:
count_positive = np.empty(len(gamma_list))

for i in range(len(gamma_list)):
    count_positive[i] = len(np.where(eigenvalues[i, 0] > 0)[0])

if np.allclose(count_positive, np.ones(len(gamma_list))) is True:
    print("At least one eigenvalue associated to the equilibrium point O has a positive real part for all values of gamma in the list.")

In [ ]:
is_complex = np.empty(len(gamma_list), dtype=bool)
count_positive = np.empty(len(gamma_list), dtype=bool)

for i in range(len(gamma_list)):
    is_complex[i] = np.any(np.iscomplex(eigenvalues[i, 1]))
    count_positive[i] = np.any(eigenvalues[i, 1].real > 0)

if np.allclose(count_positive, np.zeros(len(gamma_list))) is True:
    print("All eigenvalues associated to the equilibrium point P have negative real part for all values of gamma in the list.")
else:
    print("There are some values of gamma for which at least one eigenvalue associated to the equilibrium point P has a positive real part. These values are: ", gamma_list[np.where(count_positive)[0]])

In [ ]:
gamma_list[is_complex] # Value of \tilde{gamma} where the eigenvalues associated to the equilibrium point P become complex conjugate pairs

In [ ]:
kc = 1
a = np.pi * np.sqrt(2)
m = 1

gamma_list = np.arange(0.245, 0.25, 0.0001)

is_positive = np.empty(len(gamma_list), dtype=bool)

for j, g in enumerate(gamma_list):

    ds = System(kc, g, a, m) # Creates an instance of the system

    XP = ds.exact_equilibrium_points()[0]
    DFP = ds.Jacobian(XP)
    eigenavalues_P = la.eigvals(DFP)

    is_positive[j] = np.any(np.real(eigenavalues_P) > 0)

In [ ]:
gamma_list[is_positive] # More precise value of \widetilde{gamma}

### <span style="color:gray"><i>Bassins of attraction</i></span>

Careful: the following code is computationally expensive depending on the value of $\gamma$.

In [ ]:
kc = 1
a = np.pi * np.sqrt(2)
m = 1
gamma = 0.4
ds = System(kc, gamma, a, m)

t_max = 10 ** 4
dt = 1000

_, basin = ds.bassin_attraction(5, 0.01, t_max, dt)

We do not plot the basin here. The corresponding NumPy array can be saved, and the user can plot the basin using their preferred visualization method.

### <span style="color:gray"><i>AUTO² analysis</i></span>

In [ ]:
comparison_tol = np.array([1.e-8, 1.e-8, 1.e-8]) # Setting the tolerance for AUTO2

Warm up: we use AUTO² to verify the stability of the $3$ fixed point, including the origin, even though we don't learn much from it. We expect AUTO² to detect a Hopf bifurcation at $\gamma=\gamma_{\text{toy},1}$ (see main text).

In [ ]:
kc = 1
gamma = 1.0
a = np.pi * np.sqrt(2)
m = 1.0
ds = System(kc, gamma, np.pi * np.sqrt(2), 1)

parameters = {
    'gamma': ds.gamma,
    'a': ds.a,
    'm': ds.m}

fixed_points = {}
fixed_points[0] = ds.exact_equilibrium_points()[1]
fixed_points[1] = np.zeros(kc + 2)
fixed_points[2] = ds.exact_equilibrium_points()[0]

In [ ]:
initial_points = []
for p in fixed_points:
    initial_points.append({'parameters': parameters, 'initial_data': fixed_points[p]})

print(initial_points)

In [ ]:
b = BifurcationDiagram('pedlosky_auto_toy')
c = ConfigParser('c.pedlosky_auto_toy')
ndim = b.config_object.ndim
print(ndim)

In [ ]:
b.compute_fixed_points_diagram(initial_points, extra_comparison_parameters = ['A', 'B'], comparison_tol = 1.e-6, ICP = ['gamma'], UZR = {'gamma': list(np.arange(0.05,0.51,0.05))}, NPR = 0)

In [ ]:
b.plot_fixed_points_diagram()

We now use AUTO² to continue periodic orbits. We begin by iterating the long-term dynamics of the system for different values of $\gamma$, with sufficient precision to identify all stable periodic orbits. Avoid running this procedure unless necessary, as it is computationally expensive and generates many plots for visual inspection of the periodic orbits.

In [ ]:
kc = 1

gamma_list = np.array([0.1, 0.145, 0.165, 0.255]) # gamma_list = np.arange(0.145, 0.149, 0.001)
print(gamma_list)
n_ic = 100

a = np.pi * np.sqrt(2)
m = 1.

A0 = np.round(np.random.rand(n_ic) * 10 - 5, 2)
B0 = np.round(np.random.rand(n_ic) * 10 - 5, 2)
X0 = np.zeros((n_ic, kc + 2))

for i, (a_val, b_val) in enumerate(zip(A0, B0)):
    X0[i, 0] = a_val
    X0[i, 1] = b_val
    X0[i, 2: ] = - np.ones(kc) * a_val ** 2

t_max = 10 ** 4
dt = 10 ** (- 1)
time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)

len_truncated = 95000
solution_gamma = np.zeros((len(gamma_list), n_ic, len(time_integration) - len_truncated, kc + 2))

for i, g in enumerate(gamma_list):
    
    ds = System(kc, g, a, m)
    sol = ds.integration_system(time_integration, X0)
    solution_gamma[i] = sol[:, len_truncated:, :]

In [ ]:
generate_plots = False

In [ ]:
if generate_plots:
    for k in range(0, len(gamma_list), 1):
        print(gamma_list[k])
        n_ic = solution_gamma.shape[1]

        plt.figure(figsize=(8, 6))

        for i in range(n_ic):
            plt.plot(solution_gamma[k, i, :, 0], solution_gamma[k, i, :, 1], color='lightcoral', alpha=0.7)

        plt.xlabel(r'$A$', fontsize=12)
        plt.ylabel(r'$B$', fontsize=12)
        plt.grid(True, linestyle='-.', which='both')
        plt.title(f'Phase plane A vs B for gamma = {gamma_list[k]:.6f}')

        plt.show()

The following parameters are used for fast evaluation of the notebook and do not give good results for the bifurcation diagram, but they are useful to quickly check that the code is working and to have an idea of the shape of the branches. The parameters used in the paper are <code>NMX_choose = 1500</code> and <code>max_number_bp_detected_choose = 10</code>.

In [ ]:
NMX_choose = 500
max_number_bp_detected_choose = 2

<mark>Stable periodic orbit solution No. 1</mark>

In [ ]:
gamma = 0.1
filename, parameters = find_periodic_orbit(kc, gamma, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5','BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(4, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5','BP0', 'PD0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(5, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP10','BP0', 'PD0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 2</mark>

In [ ]:
gamma = 0.1513
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 1000, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5','BP0', 'PD0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5','BP0', 'PD0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 3</mark>

In [ ]:
gamma = 0.1524
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP10','BP0', 'PD0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 4</mark>

In [ ]:
gamma = 0.1565
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP10','BP0', 'PD0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 5</mark>

In [ ]:
gamma = 0.1692
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(4, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 6</mark>

In [ ]:
gamma = 0.1703
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(4, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 7</mark>

In [ ]:
gamma = 0.1752
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 8</mark>

In [ ]:
gamma = 0.1767
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 9</mark>

In [ ]:
gamma = 0.1845
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 10</mark>

In [ ]:
gamma = 0.1860
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 11</mark>

In [ ]:
gamma = 0.1939
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 12</mark>

In [ ]:
gamma = 0.1975
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 13</mark>

In [ ]:
gamma = 0.1995
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 14</mark>

In [ ]:
gamma = 0.2079
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 15</mark>

In [ ]:
gamma = 0.2199
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 16</mark>

In [ ]:
gamma = 0.2211
filename, parameters = find_periodic_orbit(kc, gamma, t_min_periodic = 500, epsilon = 1e-2, path = "Data/toy_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

Plotting the bifurcation diagram (first showing $\text{max} A$ versus $\gamma$ and then $L^2$-norm versus $\gamma$).

In [ ]:
b.plot_periodic_orbits_diagram(variables = (0, 2), excluded_labels = ['BP', 'UZ', 'LP', 'MX', 'EP', 'RG'], legend=False)
b.plot_periodic_orbits_diagram(variables = (0, 1), excluded_labels = ['BP', 'UZ', 'LP', 'MX', 'EP', 'RG'], legend=False)

We save all the branches (the user can also select some branches).

In [ ]:
export_branches(b, None, 0, "Data/toy_model")

### <span style="color:gray"><i>Period doubling cascade</i></span>

We here investigate the onset of chaos through the first period-doubling cascade. The code computes the Poincaré section at the plane $V = V_0$.

In [ ]:
A0 = - 1.
B0 = 1. 
X0 = np.array([[A0, B0, - A0 ** 2]])
V0 = 6.4

t_max = 5 * 10 ** 4
dt = 10 ** (- 2)
time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)

dgamma = 10 ** (-3) # Paper's value is 10 ** (-6)
gamma = np.round(np.arange(0.161, 0.166 + dgamma , dgamma), 6)
print(gamma)

save_point = []

for i, g in enumerate(gamma):
    transit = []
    ds = System(1, g, np.pi * np.sqrt(2), 1)
    n_total = len(time_integration)
    start = int(0.9 * n_total)
    sol_A = ds.integration_system(time_integration, X0)[0, start :, 0]
    sol_B = ds.integration_system(time_integration, X0)[0, start :, 1]
    sol_V = ds.integration_system(time_integration, X0)[0, start :, 2]
    test_V = g * (ds.g(1) * sol_A ** 2 - ds.h(1) * sol_V)
    
    crossings = np.where((sol_V[:-1] < V0) & (sol_V[1:] >= V0))[0]
    for j in crossings:
        if test_V[j] > 0:
            alpha = (V0 - sol_V[j]) / (sol_V[j+1] - sol_V[j])
            A_cross = sol_A[j] + alpha * (sol_A[j+1] - sol_A[j])
            B_cross = sol_B[j] + alpha * (sol_B[j+1] - sol_B[j])
            transit.append((A_cross, B_cross))

    save_point.append(transit)

In [ ]:
A_values = []
for points in save_point:
    transit_A = np.round(np.array([p[0] for p in points]), 4)
    unique_transit_A = np.unique(transit_A)
    A_values.append(unique_transit_A)

In [ ]:
G = []
Y = []

for g, arr in zip(gamma, A_values):
    for y in arr:
        if y > 0:
            G.append(g)
            Y.append(y)

plt.figure(figsize=(8, 6))
plt.plot(G, Y, '.', markersize=0.5)
plt.xlabel(r'$\gamma$')
plt.ylabel('Poincaré value')
plt.grid(True, linestyle='-.', which='both')
plt.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', grid_color='grey', grid_alpha=0.5)

plt.show()

## <span style="color:orange">4.  22-dimensional model</span>

<span><i>We now study the full $22$-model, defined as the full Pedlosky model with $k_c = 20$. As before, we provide the code for all the nonlinear analyses, including the basins of attraction and the continuation of periodic orbits with auto-AUTO. The linear stability analysis is not repeated, as it was already performed earlier. The Lyapunov exponents are presented in the last section of this notebook.</i></span>

### <span style="color:gray"><i>AUTO² analysis</i></span>

In [ ]:
comparison_tol = np.array([1.e-8, 1.e-8, 1.e-8]) # Setting the tolerance for AUTO2

Warm up: we use AUTO² to verify the stability of the $3$ fixed point, including the origin, even though we don't learn much from it. We expect nothing to be detected by AUTO² as the stability of the fixed points should remain the same for all positive values of $\gamma$.

In [ ]:
kc = 20
gamma = 0.05
a = np.pi * np.sqrt(2)
m = 1.0
ds = System(kc, gamma, a, m)

parameters = {
    'gamma': ds.gamma,
    'a': ds.a,
    'm': ds.m}

fixed_points = {}
fixed_points[0] = ds.exact_equilibrium_points()[1]
fixed_points[1] = np.zeros(kc + 2)
fixed_points[2] = ds.exact_equilibrium_points()[0]

In [ ]:
initial_points = []
for p in fixed_points:
    initial_points.append({'parameters': parameters, 'initial_data': fixed_points[p]})

print(initial_points)

In [ ]:
b = BifurcationDiagram('pedlosky_auto')
c = ConfigParser('c.pedlosky_auto')
ndim = b.config_object.ndim
print(ndim)

In [ ]:
b.compute_fixed_points_diagram(initial_points,extra_comparison_parameters = ['A', 'B'], comparison_tol = [0.] * 3, ICP = ['gamma'], UZR = {'gamma': list(np.arange(0.15,0.51,0.05))}, NPR = 0)

In [ ]:
b.plot_fixed_points_diagram()

We now use AUTO² to continue periodic orbits. We begin by iterating the long-term dynamics of the system for different values of $\gamma$, with sufficient precision to identify all stable periodic orbits. Avoid running this procedure unless necessary, as it is computationally expensive and generates many plots for visual inspection of the periodic orbits.

In [ ]:
gamma_list = np.array([0.1, 0.145, 0.165, 0.255]) # gamma_list = np.arange(0.145, 0.149, 0.001)
print(gamma_list)
n_ic = 100

a = np.pi * np.sqrt(2)
m = 1.

A0 = np.round(np.random.rand(n_ic) * 10 - 5, 2)
B0 = np.round(np.random.rand(n_ic) * 10 - 5, 2)
X0 = np.zeros((n_ic, kc + 2))

for i, (a_val, b_val) in enumerate(zip(A0, B0)):
    X0[i, 0] = a_val
    X0[i, 1] = b_val
    X0[i, 2:] = - np.ones(kc) * a_val ** 2

t_max = 10 ** 4
dt = 10 ** (- 1)
time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)

len_truncated = 97500
solution_gamma = np.zeros((len(gamma_list), n_ic, len(time_integration) - len_truncated, kc + 2))

for i, g in enumerate(gamma_list):
    
    ds = System(20, g, a, m)
    sol = ds.integration_system(time_integration, X0)
    solution_gamma[i] = sol[:, len_truncated:, :]

In [ ]:
# Avoid to run if not required

for k in range(0, len(gamma_list), 1):
    print(gamma_list[k])
    n_ic = solution_gamma.shape[1]

    plt.figure(figsize=(8, 6))

    for i in range(n_ic):
        plt.plot(solution_gamma[k, i, :, 0], solution_gamma[k, i, :, 1], color='lightcoral', alpha=0.7)

    plt.xlabel(r'$A$', fontsize=12)
    plt.ylabel(r'$B$', fontsize=12)
    plt.grid(True, linestyle='-.', which='both')
    plt.title(f'Phase plane A vs B for gamma = {gamma_list[k]:.4f}')

    plt.show()

As for the toy model, the following parameters are used for fast evaluation of the notebook and do not give good results for the bifurcation diagram, but they are useful to quickly check that the code is working and to have an idea of the shape of the branches. For the full $22$-dimensional model, the parameters used in the paper are <code>NMX_choose = 2000</code> and <code>max_number_bp_detected_choose = 10</code>.

In [ ]:
NMX_choose = 500
max_number_bp_detected_choose = 2

<mark>Stable periodic orbit solution No. 1</mark>

In [ ]:
gamma = 0.1
filename, parameters = find_periodic_orbit(kc, gamma, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5','BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(4, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5','BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(5, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP10','BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 2</mark>

In [ ]:
gamma = 0.1349
filename, parameters = find_periodic_orbit(kc, gamma, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP10','BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP10','BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(4, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP10','BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 3</mark>

In [ ]:
gamma = 0.1376
filename, parameters = find_periodic_orbit(kc, gamma, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(4, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(5, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 4</mark>

In [ ]:
gamma = 0.1468
filename, parameters = find_periodic_orbit(kc, gamma, dt_periodic = 10 ** (-2), t_max_periodic = 7000, epsilon = 1e-2, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 5</mark>

In [ ]:
gamma = 0.1616
filename, parameters = find_periodic_orbit(kc, gamma, dt_periodic = 10 ** (-2), t_max_periodic = 7000, epsilon = 1e-2, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 6</mark>

In [ ]:
gamma = 0.1637
filename, parameters = find_periodic_orbit(kc, gamma, epsilon = 1e-3, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 7</mark>

In [ ]:
gamma = 0.15013
filename, parameters = find_periodic_orbit(kc, gamma, epsilon = 1e-3, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 8</mark>

In [ ]:
gamma = 0.15107
filename, parameters = find_periodic_orbit(kc, gamma, epsilon = 1e-3, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 9</mark>

In [ ]:
gamma = 0.15618
filename, parameters = find_periodic_orbit(kc, gamma, epsilon = 1e-3, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, only_forward = True, ICP = ['gamma', 'T'], PAR = parameters, NTST = 800, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(4, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

<mark>Stable periodic orbit solution No. 10</mark>

In [ ]:
gamma = 0.15705
filename, parameters = find_periodic_orbit(kc, gamma, epsilon = 1e-3, path = "Data/full_model")

In [ ]:
b.level_reached = 1
b.add_periodic_orbit(filename, max_number_bp_detected = max_number_bp_detected_choose, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, ICP = ['gamma', 'T'], PAR = parameters, NTST = 1000, SP = ['LP2'], NPR = 10, NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(2, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

In [ ]:
b.restart_periodic_orbits_diagram(3, max_number_bp = None, extra_comparison_parameters = ['A', 'B'], comparison_tol = comparison_tol, backward_bp_continuation = True, ICP = ['gamma', 'T'], NTST = 800, SP = ['LP5', 'BP0'], NMX = NMX_choose)

Plotting the bifurcation diagram (first showing $\text{max} A$ versus $\gamma$ and then $L^2$-norm versus $\gamma$).

In [ ]:
b.plot_periodic_orbits_diagram(variables = (0, 2), excluded_labels = ['BP', 'UZ', 'LP', 'MX', 'EP', 'RG'], legend=False)
b.plot_periodic_orbits_diagram(variables = (0, 1), excluded_labels = ['BP', 'UZ', 'LP', 'MX', 'EP', 'RG'], legend=False)

We save all the branches (the user can also select some branches).

In [ ]:
export_branches(b, None, 0, "Data/full_model")

## <span style="color:orange">5.  Lyapunov exponents of the Pedlosky system</span>

<span><i>We now implement the Benettin algorithm for computing the Lyapunov spectrum of the Pedlosky system.</i></span>

In [ ]:
ds = System(1, 0.25, np.pi * np.sqrt(2), 1) # kc = 1 here to avoid long integration time

t_max = 10 ** 4
dt = 10 ** (- 1)
time_integration = np.linspace(0, t_max, int(t_max / dt) + 1)

In [ ]:
n_ic = 6

dgamma = 0.01
gamma_max = 0.5
gamma_list = np.linspace(gamma_max, 0., int(gamma_max / dgamma + 1))  
n_gamma = gamma_list.shape[0]
print("Number of gamma values:", n_gamma)
print("Gamma values:", gamma_list)

A0_1D = np.array([4, 1, 1, -1, 4, -5]) #rnd.randint(- 5, 5, n_ic)
B0_1D = np.array([-2, -1, -3, -5, 2, 5]) #rnd.randint(- 5, 5, n_ic)
print("'A' Initial conditions:", A0_1D)
print("'B' Initial conditions:", B0_1D)

A0 = np.tile(A0_1D, (n_gamma, 1)).T
B0 = np.tile(B0_1D, (n_gamma, 1)).T

X0 = np.zeros((n_ic, n_gamma, ds._kc + 2))

X0[:, :, 0] = A0
X0[:, :, 1] = B0

for i in range(2, X0.shape[-1]):
    X0[:, :, i] = - A0 ** 2

In [ ]:
solution_transient = np.zeros((n_ic, n_gamma, ds._kc + 2))

for i, gamma in enumerate(gamma_list):
    ds = System(1, gamma, np.pi * np.sqrt(2), 1)
    solution_transient[:, i] = ds.integration_system(time_integration, X0[:, i])[:, - 1]

In [ ]:
Lya_ic_gamma = Parallel(n_jobs=-1, backend="loky")(delayed(lyapunov_gamma)(gamma_list[i], solution_transient[:, i]) for i in range(len(gamma_list)))

In [ ]:
Lya_gamma = np.zeros((n_gamma, n_ic))
for i in range(n_gamma):
    for j in range(n_ic):
        Lya_gamma[i, j] = np.max(Lya_ic_gamma[i][j])

Lya_gamma = Lya_gamma.T

In [ ]:
colors = ['b', 'r', 'g', 'm', 'c']  # Different colors for each plot (reused)

fig, axes = plt.subplots(int(n_ic / 2), 2, figsize=(12, 10), sharex=True, sharey=True)  # 5 rows, 2 columns
axes = axes.flatten()

for i in range(n_ic):  # Now using all 10 subplots
    ax = axes[i]
    ax.set_xlabel(r'$\gamma$', fontsize=15)
    
    if i % 2 == 0: 
        ax.set_ylabel(r'$\lambda_\text{max}$', fontsize=15)
    
    ax.grid(True, linestyle='-.')
    ax.tick_params(axis='both', direction='in', length=6, width=1.5, labelsize='large', 
                   grid_color='grey', grid_alpha=0.5)
    
    ax.plot(gamma_list, Lya_gamma[i], marker='o', markersize=2, linestyle='-', color=colors[i % len(colors)])
    
    ax.legend([rf'$(A_0, B_0) = {int(A0_1D[i]), int(B0_1D[i])}$'])

plt.tight_layout()
plt.show()

We look where the maximum Lyapunov exponent changes sign to estimate the value of gamma where the transition to chaos occurs.

In [ ]:
test = Lya_gamma[:, ::-1]
arr = (test < 1e-2)

first_false_idx = np.full(arr.shape[0], -1)
for i, row in enumerate(arr):
    false_positions = np.where(row == False)[0]
    if false_positions.size > 0:
        first_false_idx[i] = false_positions[0]

gamma_list[::-1][first_false_idx]

We then look for the first gamma value for which the maximum Lyapunov exponent becomes negative, which indicates the transition to one of the stable fixed points.

In [ ]:
arr = (Lya_gamma < 0)

first_false_idx = np.full(arr.shape[0], -1)
for i, row in enumerate(arr):
    false_positions = np.where(row == False)[0]
    if false_positions.size > 0:
        first_false_idx[i] = false_positions[0]

gamma_list[first_false_idx]